In [1]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 21.6 MB/s eta 0:00:00


In [2]:
from PyPDF2 import PdfReader

pdf_path = '/content/LIRNEasia_Sinhala_Paper3.pdf'

def extract_text_from_pdf(pdf_path):
    text = ""
    try:
        reader = PdfReader(pdf_path)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    except Exception as e:
        text = f"Error reading PDF: {e}"
    return text

paper_content = extract_text_from_pdf(pdf_path)
print("--- PDF Content Extracted ---")
print(paper_content)
print("-----------------------------")

--- PDF Content Extracted ---
DRAFTA Corpus and Machine Learning
Models for F ake News Classification in
Sinhala
Vihanga Jayawickrama, Asanka Ranasinghe, Dimuthu C. Attanayake,
Y udhanjaya Wijeratne
LIRNEasia, 12 Balcombe Place, Colombo, Sri Lanka (yudhanjaya@lirneasia.net)
LIRNEasia is a pro-poor, pro-market think tank whose mis-
sion is catalyzing policy change through research to improve
people’s lives in the emerging Asia Pacific by facilitating
their use of hard and soft infrastructures through the use of
knowledge, information and technology .
DRAFT1
Abstract
W e present a dataset consisting of 3576 documents in Sinhala, drawn from Sri Lankan news
websites and factchecking operations, annotated as CREDIBLE, F ALSE, P AR TIAL or UN-
CER T AIN. The dataset has markers for the content of the document, the classification, the
web domain from which each document was retrieved, and the date on which the document
was published. W e also present the results of misinformation classificati

In [3]:
!git clone https://github.com/LIRNEasia/MisinformationCorpusSinhala

Cloning into 'MisinformationCorpusSinhala'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 27 (delta 11), reused 3 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 6.33 MiB | 14.10 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [6]:
import pandas as pd
import os

# List contents of the cloned directory again to confirm the file location
print(os.listdir('MisinformationCorpusSinhala'))

# Load the Corpus.csv file into a pandas DataFrame, specifying a different encoding
dataset_path = 'MisinformationCorpusSinhala/Corpus.csv'
df = pd.read_csv(dataset_path, encoding='latin1')

# Display the first few rows of the DataFrame
print("\nFirst 5 rows of the DataFrame:")
print(df.head())

# Print the shape of the DataFrame
print("\nShape of the DataFrame:")
print(df.shape)

['.git', 'Corpus.csv', 'README.md', 'Corpus.xlsx']

First 5 rows of the DataFrame:
   Unnamed: 0    X1               domain            datestamp       type  \
0        5418   652      bbc.com/sinhala  2020-04-03 00:00:00  UNCERTAIN   
1        5420  2077  gosip-lankanews.com  2020-10-07 00:00:00   CREDIBLE   
2        5422  3476         adaderana.lk  2020-11-03 00:00:00   CREDIBLE   
3        5424  2675            ravaya.lk  2020-09-19 00:00:00  UNCERTAIN   
4        5426  3635            anidda.lk  2020-10-04 00:00:00  UNCERTAIN   

                                             content  Unnamed: 6  Unnamed: 7  \
0  ??????????? ?? ?? ??? ????? 200?? ???? ???????...         NaN         NaN   
1  ????? ???????????????? ?????? ?????????? ?????...         NaN         NaN   
2  ???? ?????? ???????????? ??? ????? ??? ????? ?...         NaN         NaN   
3  ?????? ??? ???? ???????? ??????????? ????? ???...         NaN         NaN   
4  ?????? ???????????? ????????? ???? ??? ???????...        

## Explore and Preprocess Data


Performing initial data exploration, cleaning the dataset by removing irrelevant columns and filtering for specific classes, and apply Sinhala-specific text preprocessing steps to the 'content' column.


In [7]:
print("--- DataFrame Info ---")
df.info()

print("\n--- Number of Duplicate Rows ---")
print(f"Total duplicate rows: {df.duplicated().sum()}")

print("\n--- Distribution of 'type' Column ---")
print(df['type'].value_counts())

print("\n--- Sample Texts from 'content' Column ---")
for i, text in enumerate(df['content'].sample(5, random_state=42).tolist()):
    print(f"Sample {i+1}: {text[:200]}...") # Print first 200 characters of each sample

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Unnamed: 0   3000 non-null   int64  
 1   X1           3000 non-null   int64  
 2   domain       3000 non-null   object 
 3   datestamp    3000 non-null   object 
 4   type         3000 non-null   object 
 5   content      3000 non-null   object 
 6   Unnamed: 6   0 non-null      float64
 7   Unnamed: 7   0 non-null      float64
 8   Unnamed: 8   0 non-null      float64
 9   Unnamed: 9   0 non-null      float64
 10  Unnamed: 10  0 non-null      float64
 11  Unnamed: 11  0 non-null      float64
 12  Unnamed: 12  0 non-null      float64
dtypes: float64(7), int64(2), object(4)
memory usage: 304.8+ KB

--- Number of Duplicate Rows ---
Total duplicate rows: 0

--- Distribution of 'type' Column ---
type
UNCERTAIN    1887
CREDIBLE     1003
PARTIAL        83
FALSE          27
Na

In [8]:
df_processed = df[['content', 'type']].copy()
print("New DataFrame with 'content' and 'type' columns:")
print(df_processed.head())
print(f"Shape of new DataFrame: {df_processed.shape}")

New DataFrame with 'content' and 'type' columns:
                                             content       type
0  ??????????? ?? ?? ??? ????? 200?? ???? ???????...  UNCERTAIN
1  ????? ???????????????? ?????? ?????????? ?????...   CREDIBLE
2  ???? ?????? ???????????? ??? ????? ??? ????? ?...   CREDIBLE
3  ?????? ??? ???? ???????? ??????????? ????? ???...  UNCERTAIN
4  ?????? ???????????? ????????? ???? ??? ???????...  UNCERTAIN
Shape of new DataFrame: (3000, 2)


In [9]:
allowed_types = ['CREDIBLE', 'FALSE', 'UNCERTAIN']
df_filtered = df_processed[df_processed['type'].isin(allowed_types)].copy()

print("DataFrame after filtering for 'CREDIBLE', 'FALSE', or 'UNCERTAIN' types:")
print(df_filtered['type'].value_counts())
print(f"Shape of filtered DataFrame: {df_filtered.shape}")

DataFrame after filtering for 'CREDIBLE', 'FALSE', or 'UNCERTAIN' types:
type
UNCERTAIN    1887
CREDIBLE     1003
FALSE          27
Name: count, dtype: int64
Shape of filtered DataFrame: (2917, 2)


In [10]:
df_filtered['word_count'] = df_filtered['content'].apply(lambda x: len(str(x).split()))
df_filtered_min_words = df_filtered[df_filtered['word_count'] >= 20].copy()

print(f"Original shape after type filtering: {df_filtered.shape}")
print(f"Shape after removing articles with less than 20 words: {df_filtered_min_words.shape}")
print("First 5 rows of DataFrame after removing articles with less than 20 words:")
print(df_filtered_min_words.head())

Original shape after type filtering: (2917, 3)
Shape after removing articles with less than 20 words: (2879, 3)
First 5 rows of DataFrame after removing articles with less than 20 words:
                                             content       type  word_count
0  ??????????? ?? ?? ??? ????? 200?? ???? ???????...  UNCERTAIN        1567
1  ????? ???????????????? ?????? ?????????? ?????...   CREDIBLE         169
2  ???? ?????? ???????????? ??? ????? ??? ????? ?...   CREDIBLE         121
3  ?????? ??? ???? ???????? ??????????? ????? ???...  UNCERTAIN        1258
4  ?????? ???????????? ????????? ???? ??? ???????...  UNCERTAIN          84


In [11]:
import re

def clean_sinhala_text(text):
    text = str(text).lower()  # Convert to lowercase
    # Remove punctuation. Keep Sinhala characters and numbers.
    # Sinhala Unicode range is U+0D80 to U+0DFF
    text = re.sub(r'[^඀-෿0-9\s]', '', text) # Keep Sinhala unicode range (U+0D80 to U+0DFF), numbers and spaces
    text = re.sub(r'\s+', ' ', text).strip() # Replace multiple spaces with single space and strip whitespace
    return text

df_cleaned = df_filtered_min_words.copy()
df_cleaned['content'] = df_cleaned['content'].apply(clean_sinhala_text)

print("First 5 rows of DataFrame after cleaning 'content' column:")
print(df_cleaned.head())
print(f"Shape of cleaned DataFrame: {df_cleaned.shape}")

First 5 rows of DataFrame after cleaning 'content' column:
                                             content       type  word_count
0  200 26 12000 95000 8000 4000 8000 100 71000 70...  UNCERTAIN        1567
1                                                      CREDIBLE         169
2                2020 76 000 1824 11 000 76 000 2004   CREDIBLE         121
3                                     2015 1984 1980  UNCERTAIN        1258
4                                              29 26  UNCERTAIN          84
Shape of cleaned DataFrame: (2879, 3)


In [12]:
import re

# Placeholder for Sinhala stopwords. In a real application, this list
# would be more comprehensive, possibly loaded from a file or a library.
# Note: Due to potential issues with character encoding during CSV loading
# (indicated by '??????????' in previous outputs), the effectiveness of this
# stop word removal might be limited if Sinhala characters were not loaded correctly.
sinhala_stopwords = [
    "සහ", "හා", "ද", "වෙත", "මත", "තුළ", "කිසිදු", "සියලු", "අතර", "ඒ", "මෙම", "එම", "එය",
    "අපි", "ඔබ", "ඔහු", "ඇය", "ඔවුන්", "අප", "මට", "මා", "ගැන", "වෙනුවෙන්", "විසින්",
    "කරන", "කළ", "වන", "වූ", "නොව", "ඇති", "නැති", "බව", "සඳහා", "ලෙස"
]

def remove_sinhala_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in sinhala_stopwords]
    return ' '.join(filtered_words)

df_cleaned['content_no_stopwords'] = df_cleaned['content'].apply(remove_sinhala_stopwords)

print("First 5 rows of DataFrame with original cleaned content and content after stopword removal:")
print(df_cleaned[['content', 'content_no_stopwords']].head())
print(f"Shape of DataFrame after stopword removal: {df_cleaned.shape}")

First 5 rows of DataFrame with original cleaned content and content after stopword removal:
                                             content  \
0  200 26 12000 95000 8000 4000 8000 100 71000 70...   
1                                                      
2                2020 76 000 1824 11 000 76 000 2004   
3                                     2015 1984 1980   
4                                              29 26   

                                content_no_stopwords  
0  200 26 12000 95000 8000 4000 8000 100 71000 70...  
1                                                     
2                2020 76 000 1824 11 000 76 000 2004  
3                                     2015 1984 1980  
4                                              29 26  
Shape of DataFrame after stopword removal: (2879, 4)


In [13]:
import pandas as pd
import os

# Install openpyxl if not already installed
try:
    import openpyxl
    print("openpyxl is already installed.")
except ImportError:
    print("Installing openpyxl...")
    !pip install openpyxl
    print("openpyxl installed.")

# Load the Corpus.xlsx file into a pandas DataFrame
dataset_path_xlsx = 'MisinformationCorpusSinhala/Corpus.xlsx'
df_excel = pd.read_excel(dataset_path_xlsx)

# Display the first few rows of the DataFrame
print("\nFirst 5 rows of the DataFrame loaded from XLSX:")
print(df_excel.head())

# Print the shape of the DataFrame
print("\nShape of the DataFrame loaded from XLSX:")
print(df_excel.shape)

openpyxl is already installed.

First 5 rows of the DataFrame loaded from XLSX:
   Unnamed: 0    X1               domain  datestamp       type  \
0        5418   652      bbc.com/sinhala 2020-04-03  UNCERTAIN   
1        5420  2077  gosip-lankanews.com 2020-10-07   CREDIBLE   
2        5422  3476         adaderana.lk 2020-11-03   CREDIBLE   
3        5424  2675            ravaya.lk 2020-09-19  UNCERTAIN   
4        5426  3635            anidda.lk 2020-10-04  UNCERTAIN   

                                             content  
0  කොරෝනාවෛරසය මේ වන විට රටවල් 200කට අධික සංඛ්‍යා...  
1  ගම්පහ දිස්ත්‍රික්කයෙන් කොවිඩ් ආසාදිතයින් විශාල...  
2  දහම් පාසැල් ගුරුවරියන්ගේ නිල ඇඳුම් ලෙස දේශීය න...  
3  පසුගිය සති අන්ත පුවත්පත් සාකච්ඡාවකදී ජවිපෙ දේශ...  
4  ඉන්දීය අග්‍රාමාත්‍ය නරේන්ද්‍ර මෝදි සමඟ පැවැත්ව...  

Shape of the DataFrame loaded from XLSX:
(3000, 6)


In [14]:
print("--- DataFrame Info (from XLSX) ---")
df_excel.info()

print("\n--- Number of Duplicate Rows (from XLSX) ---")
print(f"Total duplicate rows: {df_excel.duplicated().sum()}")

print("\n--- Distribution of 'type' Column (from XLSX) ---")
print(df_excel['type'].value_counts())

print("\n--- Sample Texts from 'content' Column (from XLSX) ---")
for i, text in enumerate(df_excel['content'].sample(5, random_state=42).tolist()):
    print(f"Sample {i+1}: {text[:200]}...") # Print first 200 characters of each sample

--- DataFrame Info (from XLSX) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Unnamed: 0  3000 non-null   int64         
 1   X1          3000 non-null   int64         
 2   domain      3000 non-null   object        
 3   datestamp   3000 non-null   datetime64[ns]
 4   type        3000 non-null   object        
 5   content     3000 non-null   object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 140.8+ KB

--- Number of Duplicate Rows (from XLSX) ---
Total duplicate rows: 0

--- Distribution of 'type' Column (from XLSX) ---
type
UNCERTAIN    1887
CREDIBLE     1003
PARTIAL        83
False          27
Name: count, dtype: int64

--- Sample Texts from 'content' Column (from XLSX) ---
Sample 1: කොවිඞ් ආසාදිතයන් ගේ මරණ ප්‍රධාන කාණ්ඩ දෙකක් මත පදනම්ව වාර්තා ගත කරන බව සෞඛ්‍ය අමාත්‍යාංශය පවසයි.  සෘජු කෝවිඞ් ආශ්‍

In [15]:
df_processed_excel = df_excel[['content', 'type']].copy()
print("New DataFrame with 'content' and 'type' columns (from XLSX):")
print(df_processed_excel.head())
print(f"Shape of new DataFrame (from XLSX): {df_processed_excel.shape}")

New DataFrame with 'content' and 'type' columns (from XLSX):
                                             content       type
0  කොරෝනාවෛරසය මේ වන විට රටවල් 200කට අධික සංඛ්‍යා...  UNCERTAIN
1  ගම්පහ දිස්ත්‍රික්කයෙන් කොවිඩ් ආසාදිතයින් විශාල...   CREDIBLE
2  දහම් පාසැල් ගුරුවරියන්ගේ නිල ඇඳුම් ලෙස දේශීය න...   CREDIBLE
3  පසුගිය සති අන්ත පුවත්පත් සාකච්ඡාවකදී ජවිපෙ දේශ...  UNCERTAIN
4  ඉන්දීය අග්‍රාමාත්‍ය නරේන්ද්‍ර මෝදි සමඟ පැවැත්ව...  UNCERTAIN
Shape of new DataFrame (from XLSX): (3000, 2)


In [16]:
allowed_types_excel = ['CREDIBLE', 'FALSE', 'UNCERTAIN']
df_filtered_excel = df_processed_excel[df_processed_excel['type'].isin(allowed_types_excel)].copy()

print("DataFrame after filtering for 'CREDIBLE', 'FALSE', or 'UNCERTAIN' types (from XLSX):")
print(df_filtered_excel['type'].value_counts())
print(f"Shape of filtered DataFrame (from XLSX): {df_filtered_excel.shape}")

DataFrame after filtering for 'CREDIBLE', 'FALSE', or 'UNCERTAIN' types (from XLSX):
type
UNCERTAIN    1887
CREDIBLE     1003
Name: count, dtype: int64
Shape of filtered DataFrame (from XLSX): (2890, 2)


In [17]:
df_filtered_excel['word_count'] = df_filtered_excel['content'].apply(lambda x: len(str(x).split()))
df_filtered_min_words_excel = df_filtered_excel[df_filtered_excel['word_count'] >= 20].copy()

print(f"Original shape after type filtering (from XLSX): {df_filtered_excel.shape}")
print(f"Shape after removing articles with less than 20 words (from XLSX): {df_filtered_min_words_excel.shape}")
print("First 5 rows of DataFrame after removing articles with less than 20 words (from XLSX):")
print(df_filtered_min_words_excel.head())

Original shape after type filtering (from XLSX): (2890, 3)
Shape after removing articles with less than 20 words (from XLSX): (2853, 3)
First 5 rows of DataFrame after removing articles with less than 20 words (from XLSX):
                                             content       type  word_count
0  කොරෝනාවෛරසය මේ වන විට රටවල් 200කට අධික සංඛ්‍යා...  UNCERTAIN        1567
1  ගම්පහ දිස්ත්‍රික්කයෙන් කොවිඩ් ආසාදිතයින් විශාල...   CREDIBLE         169
2  දහම් පාසැල් ගුරුවරියන්ගේ නිල ඇඳුම් ලෙස දේශීය න...   CREDIBLE         121
3  පසුගිය සති අන්ත පුවත්පත් සාකච්ඡාවකදී ජවිපෙ දේශ...  UNCERTAIN        1258
4  ඉන්දීය අග්‍රාමාත්‍ය නරේන්ද්‍ර මෝදි සමඟ පැවැත්ව...  UNCERTAIN          84


In [18]:
import re

def clean_sinhala_text_excel(text):
    text = str(text).lower()  # Convert to lowercase
    # Remove punctuation. Keep Sinhala characters and numbers.
    # Sinhala Unicode range is U+0D80 to U+0DFF
    text = re.sub(r'[^\u0D80-\u0DFF0-9\s]', '', text) # Keep Sinhala unicode range, numbers and spaces
    text = re.sub(r'\s+', ' ', text).strip() # Replace multiple spaces with single space and strip whitespace
    return text

df_cleaned_excel = df_filtered_min_words_excel.copy()
df_cleaned_excel['content'] = df_cleaned_excel['content'].apply(clean_sinhala_text_excel)

print("First 5 rows of DataFrame after cleaning 'content' column (from XLSX):")
print(df_cleaned_excel.head())
print(f"Shape of cleaned DataFrame (from XLSX): {df_cleaned_excel.shape}")

First 5 rows of DataFrame after cleaning 'content' column (from XLSX):
                                             content       type  word_count
0  කොරෝනාවෛරසය මේ වන විට රටවල් 200කට අධික සංඛ්යාව...  UNCERTAIN        1567
1  ගම්පහ දිස්ත්රික්කයෙන් කොවිඩ් ආසාදිතයින් විශාල ...   CREDIBLE         169
2  දහම් පාසැල් ගුරුවරියන්ගේ නිල ඇඳුම් ලෙස දේශීය න...   CREDIBLE         121
3  පසුගිය සති අන්ත පුවත්පත් සාකච්ඡාවකදී ජවිපෙ දේශ...  UNCERTAIN        1258
4  ඉන්දීය අග්රාමාත්ය නරේන්ද්ර මෝදි සමඟ පැවැත්වූ ස...  UNCERTAIN          84
Shape of cleaned DataFrame (from XLSX): (2853, 3)


In [19]:
import re

# Placeholder for Sinhala stopwords, reusing the list from previous steps
sinhala_stopwords = [
    "සහ", "හා", "ද", "වෙත", "මත", "තුළ", "කිසිදු", "සියලු", "අතර", "ඒ", "මෙම", "එම", "එය",
    "අපි", "ඔබ", "ඔහු", "ඇය", "ඔවුන්", "අප", "මට", "මා", "ගැන", "වෙනුවෙන්", "විසින්",
    "කරන", "කළ", "වන", "වූ", "නොව", "ඇති", "නැති", "බව", "සඳහා", "ලෙස"
]

def remove_sinhala_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in sinhala_stopwords]
    return ' '.join(filtered_words)

df_cleaned_excel['content_no_stopwords_excel'] = df_cleaned_excel['content'].apply(remove_sinhala_stopwords)

print("First 5 rows of DataFrame with original cleaned content and content after stopword removal (from XLSX):")
print(df_cleaned_excel[['content', 'content_no_stopwords_excel']].head())
print(f"Shape of DataFrame after stopword removal (from XLSX): {df_cleaned_excel.shape}")

First 5 rows of DataFrame with original cleaned content and content after stopword removal (from XLSX):
                                             content  \
0  කොරෝනාවෛරසය මේ වන විට රටවල් 200කට අධික සංඛ්යාව...   
1  ගම්පහ දිස්ත්රික්කයෙන් කොවිඩ් ආසාදිතයින් විශාල ...   
2  දහම් පාසැල් ගුරුවරියන්ගේ නිල ඇඳුම් ලෙස දේශීය න...   
3  පසුගිය සති අන්ත පුවත්පත් සාකච්ඡාවකදී ජවිපෙ දේශ...   
4  ඉන්දීය අග්රාමාත්ය නරේන්ද්ර මෝදි සමඟ පැවැත්වූ ස...   

                          content_no_stopwords_excel  
0  කොරෝනාවෛරසය මේ විට රටවල් 200කට අධික සංඛ්යාවකට ...  
1  ගම්පහ දිස්ත්රික්කයෙන් කොවිඩ් ආසාදිතයින් විශාල ...  
2  දහම් පාසැල් ගුරුවරියන්ගේ නිල ඇඳුම් දේශීය නිෂ්ප...  
3  පසුගිය සති අන්ත පුවත්පත් සාකච්ඡාවකදී ජවිපෙ දේශ...  
4  ඉන්දීය අග්රාමාත්ය නරේන්ද්ර මෝදි සමඟ පැවැත්වූ ස...  
Shape of DataFrame after stopword removal (from XLSX): (2853, 4)


## Feature Extraction

### Subtask:
Convert the preprocessed Sinhala text data into numerical features using TF-IDF, and split the dataset into training and testing sets.


In [20]:
from sklearn.model_selection import train_test_split

# Separate features (X) and target (y)
X = df_cleaned_excel['content_no_stopwords_excel']
y = df_cleaned_excel['type']

# Split the dataset into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("First 5 entries of X_train:")
print(X_train.head())
print("First 5 entries of y_train:")
print(y_train.head())

Shape of X_train: (2282,)
Shape of X_test: (571,)
Shape of y_train: (2282,)
Shape of y_test: (571,)
First 5 entries of X_train:
1506    ගම්පහ ආසාදිතයෝ වැඩිදෙනෙක් මාළු වෙළෙඳාමට හවුල් ...
2852    යාපනය වැල්වෙට්ටිතුරෙයි උරික්කාඩු ප්රදේශයේ ඉකුත...
898     19 තත්වය ශ්රී ලංකාවේ සමාජ ආර්ථික ක්රියාවලියට ම...
697     කම්කරු අයිතිවාසිකම් සුරක්ෂිත කිරීම මූලික කර ගන...
2438    1 නව සෞඛ්ය සේවා අධ්යක්ෂ ජනරාල්වරයා වෛද්ය අසේල ...
Name: content_no_stopwords_excel, dtype: object
First 5 entries of y_train:
1506    UNCERTAIN
2852    UNCERTAIN
898     UNCERTAIN
697     UNCERTAIN
2438     CREDIBLE
Name: type, dtype: object


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit the vectorizer on the training data and transform both training and testing data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")

Shape of X_train_tfidf: (2282, 10989)
Shape of X_test_tfidf: (571, 10989)


## Model Training and Hyperparameter Tuning


Implement the best-performing model (Random Forests), train it on the TF-IDF transformed training data, and prepare for evaluation.


In [22]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the RandomForestClassifier with default parameters
# The paper did not specify hyperparameters, so we start with defaults.
random_forest_model = RandomForestClassifier(random_state=42)

# Train the model
random_forest_model.fit(X_train_tfidf, y_train)

print("RandomForestClassifier model trained successfully.")
# The trained model 'random_forest_model' is now stored for subsequent evaluation.

RandomForestClassifier model trained successfully.


## Model Evaluation


Evaluate the trained Random Forest model's performance using standard classification metrics such as accuracy, precision, recall, and F1-score.


In [23]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on the test set
y_pred = random_forest_model.predict(X_test_tfidf)

# Calculate and print accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

# Generate and print a detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 0.7566

Classification Report:
              precision    recall  f1-score   support

    CREDIBLE       0.75      0.43      0.54       195
   UNCERTAIN       0.76      0.93      0.83       376

    accuracy                           0.76       571
   macro avg       0.76      0.68      0.69       571
weighted avg       0.76      0.76      0.74       571

